# 04 — Model Evaluation

This notebook reviews the model-evaluation artifacts already produced by the satellite land-cover classification project. It aims to distinguish **training/sample-level performance**, **spatial cross-validation**, and **map-level outputs** without treating them as interchangeable.

The notebook is read-only: it loads existing reports and model predictions when available and does not retrain models or overwrite outputs. It searches for expected project files and reports clearly when an artifact is missing.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CANDIDATE_ROOTS = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in CANDIDATE_ROOTS if (p / "outputs").exists() and (p / "data").exists()), Path.cwd())
REPORTS = ROOT / "outputs" / "reports"
print("Repository root:", ROOT)
print("Reports directory:", REPORTS)
print("Reports directory exists:", REPORTS.exists())

Repository root: c:\Users\lenovo\Desktop\satellite-land-cover-classification
Reports directory: c:\Users\lenovo\Desktop\satellite-land-cover-classification\outputs\reports
Reports directory exists: True


## 1. Inventory available evaluation artifacts

Expected reports may include model comparison, classification-map comparison, and spatial cross-validation results. The notebook uses the files that actually exist in the current working copy.

In [2]:
candidate_files = {
    "model_comparison": REPORTS / "model_comparison.csv",
    "classification_map_comparison": REPORTS / "classification_map_comparison.csv",
    "spatial_cv_fold_metrics": REPORTS / "spatial_cv_fold_metrics.csv",
    "spatial_cv_2_group_comparison": REPORTS / "spatial_cv_2_group_comparison.csv",
    "training_set_diagnostic": REPORTS / "training_set_diagnostic.txt",
}
for label, p in candidate_files.items():
    print(f"{label:32s} {'FOUND' if p.exists() else 'not found'} — {p}")

model_comparison                 FOUND — c:\Users\lenovo\Desktop\satellite-land-cover-classification\outputs\reports\model_comparison.csv
classification_map_comparison    FOUND — c:\Users\lenovo\Desktop\satellite-land-cover-classification\outputs\reports\classification_map_comparison.csv
spatial_cv_fold_metrics          FOUND — c:\Users\lenovo\Desktop\satellite-land-cover-classification\outputs\reports\spatial_cv_fold_metrics.csv
spatial_cv_2_group_comparison    FOUND — c:\Users\lenovo\Desktop\satellite-land-cover-classification\outputs\reports\spatial_cv_2_group_comparison.csv
training_set_diagnostic          FOUND — c:\Users\lenovo\Desktop\satellite-land-cover-classification\outputs\reports\training_set_diagnostic.txt


## 2. Model comparison report

If present, this table is displayed without assuming that every metric or model was evaluated under the same protocol. Read its column names and any accompanying report notes before comparing scores.

In [3]:
model_path = candidate_files["model_comparison"]
if model_path.exists():
    model_comparison = pd.read_csv(model_path)
    print("Shape:", model_comparison.shape)
    display(model_comparison)
else:
    model_comparison = None
    print("model_comparison.csv is not available in this project copy.")

Shape: (3, 5)


,model,accuracy_mean,accuracy_std,macro_f1_mean,macro_f1_std
0,Extra Trees,0.897436,3.626189e-02,0.892698,0.028522
1,Random Forest,0.897436,3.626189e-02,0.888254,0.034683
2,Majority baseline,0.230769,2.775558e-17,0.075000,0.000000


## 3. Spatial cross-validation

Spatial folds test transfer to held-out spatial groups more directly than a random split, but small datasets can produce folds where a class is absent from the training partition. Such folds may be skipped by the evaluation script and must be reported rather than silently counted as successful or failed predictions.

In [4]:
spatial_path = candidate_files["spatial_cv_fold_metrics"]
if spatial_path.exists():
    spatial_cv = pd.read_csv(spatial_path)
    display(spatial_cv)
    if "status" in spatial_cv.columns:
        print("\nFold status counts:")
        display(spatial_cv["status"].value_counts(dropna=False).to_frame("fold_count"))
    evaluated = spatial_cv.copy()
    if "status" in evaluated.columns:
        evaluated = evaluated[evaluated["status"].astype(str).str.lower().eq("evaluated")]
    metric_cols = [c for c in ["accuracy", "macro_f1"] if c in evaluated.columns]
    if metric_cols:
        print("Summary over evaluated folds only:")
        display(evaluated[metric_cols].agg(["count", "mean", "std", "min", "max"]).T)
    if "missing_training_classes" in spatial_cv.columns:
        print("Rows with missing training classes:")
        display(spatial_cv.loc[spatial_cv["missing_training_classes"].notna() & spatial_cv["missing_training_classes"].astype(str).str.strip().ne(""),
                               [c for c in ["fold", "status", "missing_training_classes"] if c in spatial_cv.columns]])
else:
    spatial_cv = None
    print("spatial_cv_fold_metrics.csv is not available.")

,fold,training_points,validation_points,validation_classes,accuracy,macro_f1,status,missing_training_classes
0,1,27,12,4,0.916667,0.571429,evaluated,NaN
1,2,28,11,4,NaN,NaN,skipped_missing_training_class,water
2,3,30,9,3,0.444444,0.160000,evaluated,NaN
3,4,32,7,3,1.000000,0.600000,evaluated,NaN



Fold status counts:


,fold_count
status,
evaluated,3
skipped_missing_training_class,1


Summary over evaluated folds only:


,count,mean,std,min,max
accuracy,3.0,0.787037,0.299605,0.444444,1.0
macro_f1,3.0,0.443810,0.246201,0.160000,0.6


Rows with missing training classes:


,fold,status,missing_training_classes
1,2,skipped_missing_training_class,water


## 4. Optional alternate spatial split diagnostic

A separate two-group comparison may exist as a diagnostic. Because its split design and sample counts may differ from the main spatial cross-validation, do not merge the metrics into one average or compare them as if they were the same experiment.

In [5]:
two_group_path = candidate_files["spatial_cv_2_group_comparison"]
if two_group_path.exists():
    two_group = pd.read_csv(two_group_path)
    display(two_group)
else:
    two_group = None
    print("No two-group comparison report found.")

,fold,training_points,validation_points,validation_classes,accuracy,macro_f1,status,missing_training_classes
0,1,17,22,5,NaN,NaN,skipped_missing_training_class,water
1,2,22,17,4,0.764706,0.583333,evaluated,NaN


## 5. Classification-map class counts

Where available, the map-comparison report describes pixel counts for the generated classification rasters. A difference in mapped class area is not evidence that one model is more accurate; it only describes how the predictions differ.

In [6]:
map_path = candidate_files["classification_map_comparison"]
if map_path.exists():
    map_comparison = pd.read_csv(map_path)
    display(map_comparison)
    likely_count_cols = [c for c in map_comparison.columns if "pixel" in c.lower() or "count" in c.lower()]
    if len(likely_count_cols) >= 2:
        print("Pixel-count columns available for inspection:", likely_count_cols)
else:
    map_comparison = None
    print("classification_map_comparison.csv is not available.")

,class_id,class_name,random_forest_pixels,extra_trees_pixels,difference_pixels
0,1,vegetation,83751,80946,-2805
1,2,built_up,14656,26633,11977
2,3,bare_soil,27500,20600,-6900
3,4,water,1635,1561,-74
4,5,road,10835,8637,-2198


Pixel-count columns available for inspection: ['random_forest_pixels', 'extra_trees_pixels', 'difference_pixels']


## 6. Optional diagnostic text and sample-level metrics

A training-set diagnostic file may document data limitations. If available, its text is displayed below. Any high score calculated on the same small labelled sample used to fit a model should be described as sample-level/training performance unless the evaluation protocol explicitly proves otherwise.

In [7]:
diagnostic_path = candidate_files["training_set_diagnostic"]
if diagnostic_path.exists():
    print(diagnostic_path.read_text(encoding="utf-8", errors="replace"))
else:
    print("No training_set_diagnostic.txt found.")

training_csv = ROOT / "outputs" / "training_dataset.csv"
if training_csv.exists():
    training_df = pd.read_csv(training_csv)
    print("\nTraining table shape:", training_df.shape)
    label_col = "class_name" if "class_name" in training_df.columns else ("class_id" if "class_id" in training_df.columns else None)
    if label_col:
        print("Training samples per class:")
        display(training_df[label_col].fillna("MISSING").value_counts().rename_axis("class").to_frame("samples"))
else:
    print("outputs/training_dataset.csv not found.")

TRAINING-SET DIAGNOSTIC

Number of samples: 39
Accuracy: 1.0000
Macro F1-score: 1.0000

Class ID order (ID: name):
[(1, 'vegetation'), (2, 'built_up'), (3, 'bare_soil'), (4, 'water'), (5, 'road')]

Classification report:
              precision    recall  f1-score   support

  vegetation       1.00      1.00      1.00        10
    built_up       1.00      1.00      1.00         9
   bare_soil       1.00      1.00      1.00         6
       water       1.00      1.00      1.00         6
        road       1.00      1.00      1.00         8

    accuracy                           1.00        39
   macro avg       1.00      1.00      1.00        39
weighted avg       1.00      1.00      1.00        39

Confusion matrix:
Rows = actual classes; columns = predicted classes.
[[10  0  0  0  0]
 [ 0  9  0  0  0]
 [ 0  0  6  0  0]
 [ 0  0  0  6  0]
 [ 0  0  0  0  8]]


Training table shape: (39, 16)
Training samples per class:


,samples
class,
vegetation,10
built_up,9
road,8
bare_soil,6
water,6


## 7. Evaluation checklist and conclusions

Before reporting a model as reliable, confirm:

1. **Protocol clarity:** identify the training/validation split and ensure validation labels were not used to fit or tune the model.
2. **Class-wise metrics:** report precision, recall, and F1 per class where support is sufficient; overall accuracy alone can hide weak classes.
3. **Spatial generalization:** report each spatial fold, including skipped folds and missing classes. Do not present the mean as covering folds that were not evaluated.
4. **Sample size:** small and spatially clustered training samples limit confidence in metrics.
5. **Map interpretation:** pixel counts and mapped area are descriptive outputs, not accuracy measurements.
6. **Reproducibility:** record input dataset, predictors, model settings, random seed, and software versions.

For this project, previous diagnostics indicated that water samples were concentrated in one spatial group and at least one spatial fold lacked water in its training partition. Treat spatial-CV results cautiously and state the evaluated-fold count alongside any mean. These conclusions should be rechecked against the current report loaded above.

**No model is retrained and no report is overwritten by this notebook.**